Clone the SimPip repo:

In [1]:
!git clone https://$GITHUB_TOKEN@github.com/PrimozRavbar/SimPipeline.git

Cloning into 'SimPipeline'...
remote: Enumerating objects: 227, done.
remote: Counting objects: 100% (227/227), done.
remote: Compressing objects: 100% (166/166), done.
remote: Total 227 (delta 116), reused 154 (delta 57), pack-reused 0 (from 0)
Receiving objects: 100% (227/227), 8.41 MiB | 12.14 MiB/s, done.
Resolving deltas: 100% (116/116), done.


In [2]:
%cd /content/SimPipeline
!git pull

/content/SimPipeline
Already up to date.


Load the movielens data:

In [3]:
from world.external_data_loader import load_movielens

world_data = load_movielens("/content/SimPipeline/ml-100k")

print("Loader OK:", len(world_data["user_objects"]), len(world_data["movie_objects"]), len(world_data["rating_objects"]))

/content/SimPipeline/world/external_data_loader.py:27: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  ratings = pd.read_csv(


Loader OK: 943 1682 100000


In [4]:
import importlib

import ml_system.datalake
import ml_system.training_dataset
import ml_system.evaluation
import ml_system.build_recommendation_system

from ml_system.build_recommendation_system import build_recommendation_system
from ml_system.evaluation import RetrievalEvaluator, evaluate_popularity

Build a new system:

In [5]:
from ml_system.build_recommendation_system import build_recommendation_system

system = build_recommendation_system(
    movie_objects=world_data["movie_objects"],
    user_objects=world_data["user_objects"],
    user_states=world_data["user_states"],
    movie_states=world_data["movie_states"],
)

print("System build OK")

System build OK


Empty the datalake if need be:

In [ ]:
system["datalake"].events.clear()
print("DataLake events:", len(system["datalake"].events))

DataLake events: 0


# Run the simulation:

In [10]:
system["sim"].run(5000)

Steps: 5000 | Events: 21207 | RecommendationShownEvent: 10000 | RecommendationClickedEvent: 5944 | WatchEvent: 4809 | RatingEvent: 454


In [11]:
events = system["datalake"].events

from collections import Counter

counts = Counter(type(event).__name__ for event in events)

print("Total events:", len(events))
print(counts)

Total events: 21207
Counter({'RecommendationShownEvent': 10000, 'RecommendationClickedEvent': 5944, 'WatchEvent': 4809, 'RatingEvent': 454})


Run the offline feature computation for training (Spark jobs...):

In [8]:
system["spark"].run_offline_feature_job(
    system["datalake"].events,
    system["offline_pipeline"].feature_pipeline
)
print("Spark offline feature job complete.")

Spark offline feature job complete.


# Train a Two-Tower model:

In [9]:
from ml_system.retrain import retrain

system["offline_pipeline"].dataset_generator.negative_samples = 5
system["offline_pipeline"].trainer.epochs = 5

artifacts = retrain(system)

epoch: 1 loss: 3.6254513955885366
epoch: 2 loss: 3.5484346215442946
epoch: 3 loss: 3.4625443053501908
epoch: 4 loss: 3.332889300520702
epoch: 5 loss: 3.1685768532496628


Test the training quality basic metrics (Recall@20):

In [10]:
from simulation.events import RecommendationClickedEvent, WatchEvent
from simulation.events import RetrievalRequest

events = system["datalake"].events

hits = 0
evaluated = 0

for event in events:
    if not isinstance(event, (RecommendationClickedEvent, WatchEvent)):
        continue

    recommendations = system["retriever"].retrieve(
        RetrievalRequest(
            user_id=event.user_id,
            k=20
        )
    )

    evaluated += 1

    if event.movie_id in recommendations:
        hits += 1

recall_at_20 = hits / evaluated if evaluated else 0.0

print(f"Hits: {hits}")
print(f"Evaluated: {evaluated}")
print(f"Recall@20: {recall_at_20:.4%}")

Hits: 1080
Evaluated: 5561
Recall@20: 19.4210%


To improve Recall@20, train for more epochs over the same datalake.

Next, EMPTY the datalake and run another simulation (do NOT rebuild the System - keep the same model parameters).

Train with the new data in the datalake. The final Recall@20 should substentially increase (3X).

In [ ]:
from simulation.events import RecommendationClickedEvent, WatchEvent
from simulation.events import RetrievalRequest

events = system["datalake"].events

hits = 0
evaluated = 0

for event in events:
    if not isinstance(event, (RecommendationClickedEvent, WatchEvent)):
        continue

    recommendations = system["retriever"].retrieve(
        RetrievalRequest(
            user_id=event.user_id,
            k=10
        )
    )

    evaluated += 1

    if event.movie_id in recommendations:
        hits += 1

recall_at_10 = hits / evaluated if evaluated else 0.0

print(f"Hits: {hits}")
print(f"Evaluated: {evaluated}")
print(f"Recall@10: {recall_at_10:.4%}")

Hits: 2085
Evaluated: 5363
Recall@10: 38.8775%


Inspect Spark processes:

In [ ]:
spark = system["spark"]

print("Application:", spark.application)
print("Application metrics:", spark.application.metrics)
print("Scheduler:", spark.scheduler)
print("Task scheduler:", spark.task_scheduler)
print("Executor:", spark.executor)

Application: <ml_system.spark_sim.application.SparkApplication object at 0x7e797e4fd150>
Application metrics: {'jobs_submitted': 1, 'jobs_succeeded': 1, 'jobs_failed': 0}
Scheduler: <ml_system.spark_sim.scheduler.Scheduler object at 0x7e797e4fd450>
Task scheduler: <ml_system.spark_sim.task_scheduler.TaskScheduler object at 0x7e797e4fd550>
Executor: <ml_system.spark_sim.executor.Executor object at 0x7e797e4fd350>
